# 8j — Preliminary composable forecast (four ways)

Implements `inst/1a_preliminary_framework_plan.md` + the fixes/extensions in `inst/1c`,
`inst/1d`, `inst/1e`: a joint renewal / next-generation-matrix model driven by **age-pair
contact-degree distributions**, scored **four ways** — the 2×2 grid of {unweighted
**NegBin**, weighted **Hurdle-Weibull**} degree models × {**Mean**, **Neighbourhood**} NGM.

The contact **mean** is estimated **per week** with **structural reciprocity**
(`log μ_{i→j} = r + log Nⱼ`) and **separable spatio-temporal-GP smoothing** across the age-pair
grid *and over weeks* (inst/1e, §5) — each age-pair is a temporally-correlated GP sharing one
temporal length-scale `ρ_time` (matrix-normal field `η·Lp·z·Ltᵀ`), plus a decoupled
temporally-smooth level `cₜ = c + σ_c·(Lt·z_c)`, so the renewal NGM `N(t)` varies through that
week's `C*ₜ`. Forecasts use the **contact-updated iterate** over **4 origins** × 4 horizons;
**WIS** is computed on a **log scale** and aggregated **by horizon** via R `scoringutils`.

Fitting uses **Pathfinder.jl** (parsimonious fit / init) and optionally **Turing NUTS** (`USE_NUTS`).
See the specs for modelling details and the remaining lean simplifications (per-week **dispersion**
is not temporally smoothed; reduced transmission block).

> **This notebook now does FITTING ONLY.** It fits and caches the MCMC chains (the slow part). Forecast assembly, WIS scoring, and all diagnostic figures moved to `9j_forecast_diagnostics.ipynb`, which reloads these chains.

In [ ]:
ENV["GKSwstype"] = "100"   # headless GR (off-screen PNG) for nbconvert
include("forecast_utils.jl")   # single preamble: base + CoMix pipeline + forecasting framework
using Random, Statistics
mkpath("../res")

USE_NUTS = false   # true ⇒ formal Turing NUTS fit (slow); false ⇒ Pathfinder parsimonious fit

## §1 Window, infection/antibody data, and age-pair degree data

In [ ]:
# `constant_contacts = false` ⇒ contact degree estimated PER WEEK, temporally smoothed by a
# separable spatio-temporal GP (shared ρ_diag/ρ_gap/ρ_time, η, σ_c; scalar intercept c +
# temporal-level GP cₜ = c + σ_c·(Lt·z_c) + matrix-normal field η·Lp·z·Ltᵀ). The renewal NGM
# then varies in time through contacts as well as antibody: N(t) uses that week's C*ₜ.
# (Set true for the pooled one-C*-per-window preliminary.)
cfg  = FrameworkConfig(constant_contacts = false)
grid = cis_age_grid()

# Read the CoMix contact data AND the inc2prev infection/antibody estimates ONCE and reuse them
# across every window (avoids re-reading/re-joining the full Arrow and re-parsing the estimates
# CSV per origin×horizon). Then roll the forecast origin over the whole period the current
# datasets support ("available period"): each origin needs a 12-week fit/lag window back to the
# first inc2prev week, and contact data out to origin+4 for the contact-updated iterate
# (horizon-h contacts observed at t₀+h). `available_forecast_origins` derives the range.
raw  = load_raw_contact_inputs()
inf  = load_raw_infection_inputs()          # (; df, tmap) — read estimates_age_ab.csv once
FORECAST_ORIGINS = available_forecast_origins(cfg; grid = grid, craw = raw.craw)
wins = [WeeklyWindow(o; n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons)
        for o in FORECAST_ORIGINS]

println("contact data span  : ", extrema(skipmissing(raw.craw.date)))
println("forecast origins   : ", length(wins), " weekly, ",
        first(FORECAST_ORIGINS), " … ", last(FORECAST_ORIGINS))
let w = wins[1], wd0 = load_window_data(wins[1], inf.df, inf.tmap; grid = grid)
    println("origin[1] fit weeks: ", w.fit_weeks[1], " … ", w.fit_weeks[end])
    println("weekly infections @ origin[1] (age): ", round.(wd0.I_mean[:, end]; digits = 0))
end

In [ ]:
# Fit config: the four combos and the parallel-fit concurrency (CPU- and memory-balanced).
combos = [(dm, nb) for dm in (NegBinAgePair(), HurdleWeibullAgePair())
                    for nb in (MeanNGM(), NeighbourhoodDegreeNGM())]
MAX_FIT_CONCURRENCY = fit_concurrency()          # min(threads, cores−1, RAM-budget)
if Threads.nthreads() == 1
    @warn "Julia has 1 thread — pre-fit runs sequentially. Start with JULIA_NUM_THREADS>1 " *
          "(e.g. $(max(1, Sys.CPU_THREADS - 1))) for parallel fitting."
end
println("combos = ", length(combos), " | fit concurrency = ", MAX_FIT_CONCURRENCY,
        " | total fits = ", length(wins) * length(combos) * length(cfg.horizons),
        " (cached ones are skipped)")

## §2 Roll over the available period — fit four ways, forecast 1–4 weeks ahead

For **each weekly origin** across the available period the four combos are fit and forecast.
The contact **mean** is a **reciprocity-structural, GP-smoothed** field (one symmetric
log-rate per unordered age pair, separable-RBF smoothing over age midpoints 70+→74.5,
`log μ_{i→j}=r+log Nⱼ` ⟹ exact reciprocity). Forecasting is the **contact-updated iterate**
(inst/1d): per origin t₀ and horizon `h`, the degree window ends at `t₀+h` (contacts
contemporaneous with the target week; infections/antibody frozen at t₀), the NGM is refreshed
and one renewal step taken.

The fit is a **global pool with lazy prefetch** (`prefit_chains_streaming!`): every
`(origin × combo × horizon)` chain is fit under one `MAX_FIT_CONCURRENCY` cap that stays
saturated **across origin boundaries** — as soon as a fit slot frees it is taken by the next
origin's chains, so the CPU never drains at an origin boundary and there is a single global
warmup (not one per origin). Each origin's window + 4 degree windows are built **lazily** by a
background producer running a couple of origins ahead of the fitting frontier (reusing the single
raw contact/estimate reads) and freed once that origin's last chain completes, so the
single-threaded DataFrames prep overlaps the parallel MCMC and memory stays **bounded** (~2–3
origins' data). Chains are cached per (origin, horizon) under `../dt_intermediate/8j_chn_*.jld2`,
so the run is **resumable** — a re-run reloads finished chains and only fits what's missing,
byte-identical to the serial roll.

In [ ]:
# Parallel, resumable PRE-FIT ONLY — GLOBAL fit pool with lazy per-origin dataset prefetch.
# All (origin × combo × horizon) chains are fit under ONE concurrency cap that stays saturated
# ACROSS origin boundaries: a freed fit slot is immediately taken by the next origin's chains
# (no per-origin barrier / drain tail / per-origin warmup). Each origin's window data + 4
# contact/degree windows are built lazily by a single producer that runs a couple of origins
# ahead of the fitting frontier and frees them once that origin's last chain completes, so the
# single-threaded DataFrames prep overlaps the parallel MCMC and memory stays bounded. Chains are
# cached to ../dt_intermediate/8j_chn_<degree>_<ngm>_<contacts>_<origin>_h<h>.jld2; cached chains
# are skipped (resumable), and results are byte-identical to the serial loop. Forecast assembly,
# scoring, and diagnostics live in 9j_forecast_diagnostics.ipynb (it reloads these chains — run this first).

# One origin's datasets: window infection/antibody + the 4 contact/degree windows. Pure &
# deterministic given the shared read-only `inf`/`raw` reads, so it is safe to call from the
# background prefetch producer.
build_origin_data(win_o) = (
    load_window_data(win_o, inf.df, inf.tmap; grid = grid),                     # reuse the single CSV read
    [prepare_degree_data(
         WeeklyWindow(win_o.origin + Day(7 * h);
                      n_fit = cfg.n_fit, smax = cfg.smax, horizons = cfg.horizons),
         cfg; grid = grid, setting = :all,
         df_part_raw = raw.df_part, craw_raw = raw.craw)                        # reuse the single Arrow read
     for h in cfg.horizons])

t0  = time()
res = prefit_chains_streaming!(combos, wins, cfg;
          data_provider = (oi, win_o) -> build_origin_data(win_o),
          grid = grid, setting = :all, use_nuts = USE_NUTS,
          save_dir = "../dt_intermediate", max_concurrent = MAX_FIT_CONCURRENCY)
println("streaming pre-fit: ", res.fitted, " fitted / ", res.requested, " requested, ",
        res.failed, " failed in ", round(Int, time() - t0), "s")

# Verifiable tail: count how many of this run's chain-cache files are present.
chain_paths = ["../dt_intermediate/8j_chn_$(degree_label(dm))_$(ngm_label(nb))_" *
               "$(contacts_label(cfg))_$(win.origin)_h$(h).jld2"
               for win in wins, (dm, nb) in combos, h in cfg.horizons]
n_have, n_expect = count(isfile, chain_paths), length(chain_paths)
println("cached chains: $n_have / $n_expect present under ../dt_intermediate ",
        "(", contacts_label(cfg), " contacts) — feed 9j_forecast_diagnostics.ipynb")